In [ ]:
!pip install -q transformers sentencepiece x-transformers optuna scikit-learn

# WangchanBERTa + x-transformers สำหรับ Prachatai multi-label 12 คลาส

Notebook นี้ดัดแปลงเฉพาะส่วน Dataset จาก SIED-Thai ให้ใช้ไฟล์ Prachatai ที่แบ่ง `train`, `validation` และ `test` มาแล้ว เพื่อเก็บผลการทดลองเพิ่มโดยไม่เปลี่ยนสถาปัตยกรรมหรือขั้นตอนฝึกของโมเดลต้นฉบับ

- Text column: `body_text`
- Labels: 12 หมวดข่าว Prachatai
- Loss: `BCEWithLogitsLoss`
- Prediction: `sigmoid + tuned threshold`
- Encoder: WangchanBERTa frozen
- Decoder/classifier: x-transformers ที่ฝึกและ optimize ด้วย Optuna
- Dataset split: ใช้ official Train/Validation/Test โดยตรง ไม่ split ใหม่ ไม่ลบข้อมูลซ้ำ และไม่ตัดแถว all-zero
- Training: 100 epochs ทุก Optuna trial และ 100 epochs สำหรับ Final model ไม่มี early stopping


In [ ]:
import os
import random
import shutil
import subprocess
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from tqdm import tqdm

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModel,
    get_linear_schedule_with_warmup
)

from x_transformers import Decoder

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    precision_score,
    recall_score,
    classification_report,
    hamming_loss
)

import optuna


In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

In [ ]:
MODEL_NAME = "airesearch/wangchanberta-base-att-spm-uncased"

MAX_LEN = 128

EPOCHS = 100

BATCH_SIZE = 64

NUM_LABELS = 12

N_TRIALS = 10

LABEL_COLUMNS = [
    "politics",
    "human_rights",
    "quality_of_life",
    "international",
    "social",
    "environment",
    "economics",
    "culture",
    "labor",
    "national_security",
    "ict",
    "education"
]

TEXT_COLUMN = "body_text"

EXPECTED_FILES = {
    "train": "prachatai_train.csv",
    "validation": "prachatai_validation.csv",
    "test": "prachatai_test.csv"
}

SEARCH_ROOTS = [
    Path("/content"),
    Path("/content/prachatai_data"),
    Path("/mnt/data")
]

ARCHIVE_CANDIDATES = [
    Path("/content/Prachatai.rar"),
    Path("/mnt/data/Prachatai.rar")
]

EXTRACT_DIR = Path("/content/prachatai_data")


def find_dataset_paths():
    paths = {}

    for split, filename in EXPECTED_FILES.items():
        candidates = []

        for root in SEARCH_ROOTS:
            if root.exists():
                candidates.extend(root.rglob(filename))

        if candidates:
            paths[split] = str(candidates[0])

    return paths


def extract_prachatai_archive(archive_path):
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)

    if not shutil.which("unrar") and not shutil.which("unar"):
        subprocess.run(["apt-get", "update", "-qq"], check=True)

        install_unrar = subprocess.run(
            ["apt-get", "install", "-y", "-qq", "unrar"],
            check=False
        )

        if install_unrar.returncode != 0:
            subprocess.run(
                ["apt-get", "install", "-y", "-qq", "unar"],
                check=True
            )

    if shutil.which("unrar"):
        command = [
            "unrar", "x", "-o+",
            str(archive_path),
            str(EXTRACT_DIR)
        ]
    elif shutil.which("unar"):
        command = [
            "unar", "-f", "-o",
            str(EXTRACT_DIR),
            str(archive_path)
        ]
    else:
        raise RuntimeError("ไม่พบโปรแกรมสำหรับแตกไฟล์ RAR")

    subprocess.run(command, check=True)


DATASET_PATHS = find_dataset_paths()

if len(DATASET_PATHS) != 3:
    archive_path = next(
        (path for path in ARCHIVE_CANDIDATES if path.exists()),
        None
    )

    if archive_path is None:
        raise FileNotFoundError(
            "กรุณาอัปโหลด Prachatai.rar หรือ CSV ทั้ง 3 ไฟล์ไปยัง /content"
        )

    print("EXTRACTING:", archive_path)
    extract_prachatai_archive(archive_path)
    DATASET_PATHS = find_dataset_paths()

if len(DATASET_PATHS) != 3:
    raise FileNotFoundError(
        f"พบไฟล์ Prachatai ไม่ครบ: {DATASET_PATHS}"
    )

TRAIN_CSV_PATH = DATASET_PATHS["train"]
VALIDATION_CSV_PATH = DATASET_PATHS["validation"]
TEST_CSV_PATH = DATASET_PATHS["test"]

print("TRAIN CSV     :", TRAIN_CSV_PATH)
print("VALIDATION CSV:", VALIDATION_CSV_PATH)
print("TEST CSV      :", TEST_CSV_PATH)


In [ ]:
required_columns = [TEXT_COLUMN] + LABEL_COLUMNS


def load_prachatai_split(path, split_name):
    df = pd.read_csv(
        path,
        usecols=required_columns,
        low_memory=False
    )

    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"{split_name} ไม่มี column: {col}")

    df = df[required_columns].copy()
    df[TEXT_COLUMN] = df[TEXT_COLUMN].astype(str).str.strip()

    for col in LABEL_COLUMNS:
        df[col] = pd.to_numeric(
            df[col],
            errors="coerce"
        ).fillna(0).astype(int)

        invalid_values = ~df[col].isin([0, 1])

        if invalid_values.any():
            values = df.loc[invalid_values, col].unique().tolist()
            raise ValueError(
                f"{split_name}/{col} มีค่านอกเหนือจาก 0/1: {values}"
            )

    return df.reset_index(drop=True)


train_df = load_prachatai_split(
    TRAIN_CSV_PATH,
    "train"
)

val_df = load_prachatai_split(
    VALIDATION_CSV_PATH,
    "validation"
)

test_df = load_prachatai_split(
    TEST_CSV_PATH,
    "test"
)

print("TRAIN     :", len(train_df))
print("VALIDATION:", len(val_df))
print("TEST      :", len(test_df))


In [ ]:
# ใช้ official split จาก Dataset โดยตรง ไม่มีการ split หรือย้ายข้อมูลข้ามชุด

print("TRAIN LABEL DISTRIBUTION")
print(train_df[LABEL_COLUMNS].sum())

print("\nVALIDATION LABEL DISTRIBUTION")
print(val_df[LABEL_COLUMNS].sum())

print("\nTEST LABEL DISTRIBUTION")
print(test_df[LABEL_COLUMNS].sum())

print("\nLABEL CARDINALITY")

print("TRAIN")
print(
    train_df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

print("\nVALIDATION")
print(
    val_df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)

print("\nTEST")
print(
    test_df[LABEL_COLUMNS]
    .sum(axis=1)
    .value_counts()
    .sort_index()
)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [ ]:
class EmotionDataset(Dataset):

    def __init__(self, df):

        self.texts = df[TEXT_COLUMN].tolist()

        self.labels = df[LABEL_COLUMNS].values.astype(np.float32)

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts[idx])

        labels = self.labels[idx]

        encoding = tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.float)
        }
train_dataset = EmotionDataset(train_df)

val_dataset = EmotionDataset(val_df)

test_dataset = EmotionDataset(test_df)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

In [ ]:
# pos_weight สำหรับ BCEWithLogitsLoss
# สูตรที่ใช้: negative / positive
# เหมาะกับ multi-label มากกว่า total / positive

label_counts = train_df[LABEL_COLUMNS].sum().values.astype(np.float32)

total_samples = len(train_df)

neg_counts = total_samples - label_counts

pos_weights = neg_counts / (label_counts + 1e-6)

pos_weights = torch.tensor(
    pos_weights,
    dtype=torch.float
).to(device)

print("LABEL COUNTS:", label_counts)
print("POS WEIGHTS :", pos_weights)


In [ ]:
class AttentionPooling(nn.Module):

    def __init__(self, hidden_size):

        super().__init__()

        self.attention = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, x, mask):

        scores = self.attention(x).squeeze(-1)

        scores = scores.masked_fill(mask == 0, -1e9)

        weights = torch.softmax(scores, dim=1)

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled

In [ ]:
class EmotionModel(nn.Module):

    def __init__(
        self,
        depth=1,
        heads=2,
        attn_dropout=0.2,
        ff_dropout=0.2,
        ff_mult=2
    ):

        super().__init__()

        self.encoder = AutoModel.from_pretrained(
            MODEL_NAME
        )

        # Fixed Thai encoder:
        # freeze WangchanBERTa แล้ว train เฉพาะ decoder + pooling + classifier
        for param in self.encoder.parameters():
            param.requires_grad = False

        hidden_size = self.encoder.config.hidden_size

        self.decoder = Decoder(
            dim=hidden_size,
            depth=depth,
            heads=heads,
            attn_dropout=attn_dropout,
            ff_dropout=ff_dropout,
            ff_mult=ff_mult
        )

        self.pooling = AttentionPooling(hidden_size)

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(
            hidden_size,
            NUM_LABELS
        )

    def forward(self, input_ids, attention_mask):

        # Encoder ถูก freeze จึงไม่ต้องเก็บ gradient
        with torch.no_grad():
            outputs = self.encoder(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        x = outputs.last_hidden_state

        x = self.decoder(x)

        x = self.pooling(x, attention_mask)

        x = self.dropout(x)

        logits = self.fc(x)

        return logits


In [ ]:
def optimize_thresholds(y_true, y_probs):

    best_thresholds = []

    for i in range(NUM_LABELS):

        best_thr = 0.5
        best_f1 = 0

        for thr in np.arange(0.1, 0.9, 0.05):

            preds = (y_probs[:, i] >= thr).astype(int)

            score = f1_score(
                y_true[:, i],
                preds,
                zero_division=0
            )

            if score > best_f1:

                best_f1 = score

                best_thr = thr

        best_thresholds.append(best_thr)

    return np.array(best_thresholds)

In [ ]:
def evaluate(model, loader, thresholds=None):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            logits = model(
                input_ids,
                attention_mask
            )

            probs = torch.sigmoid(logits)

            all_probs.append(
                probs.cpu().numpy()
            )

            all_labels.append(
                labels.cpu().numpy()
            )

    all_probs = np.vstack(all_probs)

    all_labels = np.vstack(all_labels)

    if thresholds is None:

        thresholds = np.array(
            [0.5] * NUM_LABELS
        )

    preds = (
        all_probs >= thresholds
    ).astype(int)

    micro_f1 = f1_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_f1 = f1_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        preds,
        average="weighted",
        zero_division=0
    )

    micro_precision = precision_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_precision = precision_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    micro_recall = recall_score(
        all_labels,
        preds,
        average="micro",
        zero_division=0
    )

    macro_recall = recall_score(
        all_labels,
        preds,
        average="macro",
        zero_division=0
    )

    # สำหรับ multi-label accuracy_score คือ subset accuracy / exact match
    subset_accuracy = accuracy_score(
        all_labels,
        preds
    )

    h_loss = hamming_loss(
        all_labels,
        preds
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "micro_precision": micro_precision,
        "macro_precision": macro_precision,
        "micro_recall": micro_recall,
        "macro_recall": macro_recall,
        "accuracy": subset_accuracy,
        "subset_accuracy": subset_accuracy,
        "hamming_loss": h_loss,
        "labels": all_labels,
        "preds": preds,
        "probs": all_probs
    }


In [ ]:
def train_model(
    model,
    train_loader,
    val_loader,
    lr,
    weight_decay
):

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weights
    )

    # train เฉพาะ parameter ที่ requires_grad=True
    # encoder ถูก freeze แล้ว จึง update เฉพาะ decoder + pooling + classifier
    trainable_params = [p for p in model.parameters() if p.requires_grad]

    optimizer = torch.optim.AdamW(
        trainable_params,
        lr=lr,
        weight_decay=weight_decay
    )

    total_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(
            0.1 * total_steps
        ),
        num_training_steps=total_steps
    )

    use_amp = device.type == "cuda"

    scaler = torch.cuda.amp.GradScaler(
        enabled=use_amp
    )

    for epoch in range(EPOCHS):

        model.train()

        total_loss = 0

        loop = tqdm(train_loader)

        for batch in loop:

            input_ids = batch["input_ids"].to(device)

            attention_mask = batch["attention_mask"].to(device)

            labels = batch["labels"].to(device)

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(
                enabled=use_amp
            ):

                logits = model(
                    input_ids,
                    attention_mask
                )

                loss = criterion(
                    logits,
                    labels
                )

            scaler.scale(loss).backward()

            torch.nn.utils.clip_grad_norm_(
                trainable_params,
                1.0
            )

            scaler.step(optimizer)

            scaler.update()

            scheduler.step()

            total_loss += loss.item()

            loop.set_description(
                f"Epoch {epoch+1}/{EPOCHS}"
            )

            loop.set_postfix(
                loss=loss.item()
            )

        val_result = evaluate(
            model,
            val_loader
        )

        print("\n======================")
        print(f"Epoch {epoch+1}")
        print("======================")

        print(
            f"VAL SUBSET ACC : {val_result['subset_accuracy']:.4f}"
        )

        print(
            f"VAL MICRO P/R/F1: "
            f"{val_result['micro_precision']:.4f} / "
            f"{val_result['micro_recall']:.4f} / "
            f"{val_result['micro_f1']:.4f}"
        )

        print(
            f"VAL MACRO P/R/F1: "
            f"{val_result['macro_precision']:.4f} / "
            f"{val_result['macro_recall']:.4f} / "
            f"{val_result['macro_f1']:.4f}"
        )

        print(
            f"VAL WEIGHTED F1: {val_result['weighted_f1']:.4f}"
        )

        print(
            f"VAL HAMMING LOSS: {val_result['hamming_loss']:.4f}"
        )

    return model


In [ ]:
def objective(trial):

    depth = trial.suggest_int(
        "depth",
        1,
        2
    )

    heads = trial.suggest_categorical(
        "heads",
        [2, 4]
    )

    attn_dropout = trial.suggest_float(
        "attn_dropout",
        0.2,
        0.5
    )

    ff_dropout = trial.suggest_float(
        "ff_dropout",
        0.2,
        0.5
    )

    ff_mult = trial.suggest_int(
        "ff_mult",
        2,
        4
    )

    lr = trial.suggest_float(
        "lr",
        1e-5,
        2e-4,
        log=True
    )

    weight_decay = trial.suggest_float(
        "weight_decay",
        1e-4,
        1e-2,
        log=True
    )

    model = EmotionModel(
        depth=depth,
        heads=heads,
        attn_dropout=attn_dropout,
        ff_dropout=ff_dropout,
        ff_mult=ff_mult
    ).to(device)

    model = train_model(
        model,
        train_loader,
        val_loader,
        lr,
        weight_decay
    )

    val_result = evaluate(
        model,
        val_loader
    )

    best_thresholds = optimize_thresholds(
        val_result["labels"],
        val_result["probs"]
    )

    val_result = evaluate(
        model,
        val_loader,
        best_thresholds
    )

    return val_result["macro_f1"]

In [ ]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=N_TRIALS
)


In [ ]:
print("\n======================")
print("BEST PARAMETERS")
print("======================")

print(study.best_params)

best_params = study.best_params

In [ ]:
final_model = EmotionModel(
    depth=best_params["depth"],
    heads=best_params["heads"],
    attn_dropout=best_params["attn_dropout"],
    ff_dropout=best_params["ff_dropout"],
    ff_mult=best_params["ff_mult"]
).to(device)

In [ ]:
final_model = train_model(
    final_model,
    train_loader,
    val_loader,
    best_params["lr"],
    best_params["weight_decay"]
)

In [ ]:
val_result = evaluate(
    final_model,
    val_loader
)

best_thresholds = optimize_thresholds(
    val_result["labels"],
    val_result["probs"]
)

print("\n======================")
print("BEST THRESHOLDS")
print("======================")

for label, thr in zip(
    LABEL_COLUMNS,
    best_thresholds
):
    print(f"{label}: {thr:.2f}")

In [ ]:
test_result = evaluate(
    final_model,
    test_loader,
    best_thresholds
)

print("\n======================")
print("FINAL RESULT ON OFFICIAL PRACHATAI TEST SET")
print("======================")

print(
    f"Subset Accuracy / Exact Match : {test_result['subset_accuracy']:.4f}"
)

print(
    f"Micro Precision              : {test_result['micro_precision']:.4f}"
)

print(
    f"Micro Recall                 : {test_result['micro_recall']:.4f}"
)

print(
    f"Micro F1                     : {test_result['micro_f1']:.4f}"
)

print(
    f"Macro Precision              : {test_result['macro_precision']:.4f}"
)

print(
    f"Macro Recall                 : {test_result['macro_recall']:.4f}"
)

print(
    f"Macro F1                     : {test_result['macro_f1']:.4f}"
)

print(
    f"Weighted F1                  : {test_result['weighted_f1']:.4f}"
)

print(
    f"Hamming Loss                 : {test_result['hamming_loss']:.4f}"
)


In [ ]:
print("\n======================")
print("CLASSIFICATION REPORT")
print("======================")

print(
    classification_report(
        test_result["labels"],
        test_result["preds"],
        target_names=LABEL_COLUMNS,
        zero_division=0
    )
)

In [ ]:
SAVE_PATH = "/content/final_prachatai_wangchanberta_xtransformers.pt"

torch.save(
    {
        "model_state_dict": final_model.state_dict(),
        "thresholds": best_thresholds,
        "best_params": best_params,
        "labels": LABEL_COLUMNS,
        "model_name": MODEL_NAME,
        "max_len": MAX_LEN,
        "note": "Prachatai official Train/Validation/Test; model and training logic unchanged; encoder frozen."
    },
    SAVE_PATH
)

print("\nMODEL SAVED:", SAVE_PATH)
